In [1]:
!pip install ohmeow-blurr -q
!pip install datasets -q
!pip install bert-score -q

In [2]:
import datasets
import pandas as pd
from fastai.text.all import *
from transformers import *

from blurr.data.all import *
from blurr.modeling.all import *

In [3]:
import zipfile
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)

zip_ref = zipfile.ZipFile("/content/drive/My Drive/cnndm.zip", 'r')
zip_ref.extractall("/content")
zip_ref.close()

Mounted at /content/drive/


## Data Preparation

In [4]:
train = pd.read_csv('/content/cnndm/val.txt.src', sep='\n', header=None)
train.columns=['article']
train

,article
0,"-lrb- cnn -rrb- the only thing crazier than a guy in snowbound massachusetts boxing up the powdery white stuff and offering it for sale online ? people are actually buying it . for $ 89 , self-styled entrepreneur kyle waring will ship you 6 pounds of boston-area snow in an insulated styrofoam box -- enough for 10 to 15 snowballs , he says . but not if you live in new england or surrounding states . `` we will not ship snow to any states in the northeast ! '' says waring 's website , shipsnowyo.com . `` we 're in the business of expunging snow ! '' his website and social media accounts clai..."
1,"-lrb- cnn -rrb- on the 6th of april 1996 , san jose clash and dc united strode out in front of 31,683 expectant fans at the spartan stadium in san jose , california . the historic occasion was the first ever major league soccer match -- a brave new dawn for the world 's favorite sport in a land its charms had yet to conquer . summarizing the action for espn , commentator ty keough eagerly described the momentous `` birth of a new era for american soccer . '' looking back at footage from that balmy evening now it 's hard not to feel a certain nostalgia . baggy shirts , questionable hairstyl..."
2,"-lrb- cnn -rrb- french striker bafetimbi gomis , who has a history of fainting , said he is now `` feeling well '' after collapsing during swansea 's 3-2 loss at tottenham in the premier league on wednesday . the worrying incident occurred in the first half at white hart lane -- after tottenham scored in the seventh minute -- but the 29-year-old left the pitch conscious following about five minutes of treatment . the guardian added that he was wearing an oxygen mask . play was temporarily stopped before resuming . as the match progressed , swansea tweeted that gomis was `` fine , '' with m..."
3,"-lrb- cnn -rrb- my vote for father of the year goes to curt schilling . the former major league baseball pitcher recently fired off a series of fastballs and mowed down a group of twitter trolls who made the mistake of tweeting vulgar and sexually-explicit comments about schilling 's teenage daughter . the drama started , innocently enough , on february 25 , when schilling played the role of a proud father . he sent a tweet congratulating his daughter , gabby , on being accepted to salve regina university , where she 'll play softball . it read : `` congrats to gabby schilling who will pit..."
4,"-lrb- cnn -rrb- it was an act of frustration perhaps more commonly associated with golf 's fictional anti-hero happy gilmore than the world 's reigning no 1 . player . but when rory mcilroy pulled his second shot on the eighth hole of the wgc cadillac championship into a lake friday , he might as well have been channeling the much loved adam sandler character . before continuing his round with a dropped ball , the four-time major winner launched the 3-iron used to play the offending shot into the water as well . '' -lrb- it -rrb- felt good at the time , '' a rueful mcilroy later said of th..."
...,...
4995,"a swedish football team has told of how they narrowly escaped death after a last-minute decision against booking their players on germanwings flight 4u9525 . four lufthansa subsidiary flights left barcelona around the same time this morning , division 1 team dalkurd ff , from borlänge , central sweden , had players on three of them . the team had initially planned to travel on flight 4u9525 to düsseldorf , but instead split the team over the three remaining flights from barcelona , via germany . scroll down for video . lucky escape : swedish division 1 team dalkurd ff had initially planned..."
4996,"england manager roy hodgson fears he may have to play a vastly weakened side in next tuesday 's friendly against italy , with more players set to drop out of his squad once the euro 2016 qualifier against lithuania has been played . daniel sturridge , adam lallana , fraser forster and luke shaw have all been forced to withdraw through injury alread

In [5]:
df1 = pd.read_csv('/content/cnndm/val.txt.tgt.tagged', sep='\n', header=None)
df1

,0
0,"<t> a man in suburban boston is selling snow online to customers in warmer states . </t> <t> for $ 89 , he will ship 6 pounds of snow in an insulated styrofoam box . </t>"
1,<t> the 20th mls season begins this weekend . </t> <t> league has changed dramatically since its inception in 1996 . </t> <t> some question whether rules regarding salary caps and transfers need to change . </t>
2,<t> bafetimbi gomis collapses within 10 minutes of kickoff at tottenham . </t> <t> but he reportedly left the pitch conscious and wearing an oxygen mask . </t> <t> gomis later said that he was `` feeling well '' </t> <t> the incident came three years after fabrice muamba collapsed at white hart lane . </t>
3,"<t> ruben navarrette : schilling deserves praise for taking on online haters for offensive comments about his daughter . </t> <t> navarrette : in protecting his child , schilling set a model for parenting and taught us a lesson about social media . </t>"
4,<t> rory mcilroy throws club into water at wgc cadillac championship . </t> <t> northern irishman frustrated after pulling shot into water hazard . </t>
...,...
4995,"<t> dalkurd ff from sweden were supposed to have travelled on flight 9525 . </t> <t> connection time in dusseldorf was too long , so the team split up . </t> <t> four lufthansa flights left around same time , dalkurd were on three . </t> <t> the fourth flight crashed into the alps , killing all 150 on board . </t> <t> team has spoken of shock as they ` checked in next to the victims ' </t> <t> spanish woman whose husband died on flight was ` too pregnant ' to fly . </t>"
4996,"<t> roy hodgson thinks england will lose more players to injury soon . </t> <t> he has identified one or two likely to pull out ahead of italy friendly . </t> <t> daniel sturridge , adam lallana , fraser forster and luke shaw have all already pulled out with other players struggling during training . </t> <t> england face lithuania before flying to turin for tuesday 's game . </t>"
4997,<t> daniel sturridge withdrew from the england squad on monday night . </t> <t> harry kane has scored 29 goals in all competitions for tottenham this term . </t> <t> england host lithuania in their euro 2016 qualifier on friday . </t>
4998,<t> scotland face northern ireland on wednesday and gibraltar on sunday . </t> <t> steven naismith said gordon strachan 's side is ` in it together ' now . </t> <t> everton forward said previous national call-ups brought mixed emotions . </t> <t> naismith to share hampden park with meet former rangers team-mates steven davis and kyle lafferty . </t>


In [6]:
train = pd.read_csv('/content/cnndm/val.txt.src', sep='\n', header=None)
train.columns=['article']
df1 = pd.read_csv('/content/cnndm/val.txt.tgt.tagged', sep='\n', header=None)
train['highlights'] = df1[0]
train['highlights'] = train['highlights'].apply(lambda s : re.sub('<t>', ' ', s))
train['highlights'] = train['highlights'].apply(lambda s : re.sub('</t>', ' ', s))
train.head()

,article,highlights
0,"-lrb- cnn -rrb- the only thing crazier than a guy in snowbound massachusetts boxing up the powdery white stuff and offering it for sale online ? people are actually buying it . for $ 89 , self-styled entrepreneur kyle waring will ship you 6 pounds of boston-area snow in an insulated styrofoam box -- enough for 10 to 15 snowballs , he says . but not if you live in new england or surrounding states . `` we will not ship snow to any states in the northeast ! '' says waring 's website , shipsnowyo.com . `` we 're in the business of expunging snow ! '' his website and social media accounts clai...","a man in suburban boston is selling snow online to customers in warmer states . for $ 89 , he will ship 6 pounds of snow in an insulated styrofoam box ."
1,"-lrb- cnn -rrb- on the 6th of april 1996 , san jose clash and dc united strode out in front of 31,683 expectant fans at the spartan stadium in san jose , california . the historic occasion was the first ever major league soccer match -- a brave new dawn for the world 's favorite sport in a land its charms had yet to conquer . summarizing the action for espn , commentator ty keough eagerly described the momentous `` birth of a new era for american soccer . '' looking back at footage from that balmy evening now it 's hard not to feel a certain nostalgia . baggy shirts , questionable hairstyl...",the 20th mls season begins this weekend . league has changed dramatically since its inception in 1996 . some question whether rules regarding salary caps and transfers need to change .
2,"-lrb- cnn -rrb- french striker bafetimbi gomis , who has a history of fainting , said he is now `` feeling well '' after collapsing during swansea 's 3-2 loss at tottenham in the premier league on wednesday . the worrying incident occurred in the first half at white hart lane -- after tottenham scored in the seventh minute -- but the 29-year-old left the pitch conscious following about five minutes of treatment . the guardian added that he was wearing an oxygen mask . play was temporarily stopped before resuming . as the match progressed , swansea tweeted that gomis was `` fine , '' with m...",bafetimbi gomis collapses within 10 minutes of kickoff at tottenham . but he reportedly left the pitch conscious and wearing an oxygen mask . gomis later said that he was `` feeling well '' the incident came three years after fabrice muamba collapsed at white hart lane .
3,"-lrb- cnn -rrb- my vote for father of the year goes to curt schilling . the former major league baseball pitcher recently fired off a series of fastballs and mowed down a group of twitter trolls who made the mistake of tweeting vulgar and sexually-explicit comments about schilling 's teenage daughter . the drama started , innocently enough , on february 25 , when schilling played the role of a proud father . he sent a tweet congratulating his daughter , gabby , on being accepted to salve regina university , where she 'll play softball . it read : `` congrats to gabby schilling who will pit...","ruben navarrette : schilling deserves praise for taking on online haters for offensive comments about his daughter . navarrette : in protecting his child , schilling set a model for parenting and taught us a lesson about social media ."
4,"-lrb- cnn -rrb- it was an act of frustration perhaps more commonly associated with golf 's fictional anti-hero happy gilmore than the world 's reigning no 1 . player . but when rory mcilroy pulled his second shot on the eighth hole of the wgc cadillac championship into a lake friday , he might as well have been channeling the much loved adam sandler character . before continuing his round with a dropped ball , the four-time major winner launched the 3-iron used to play the offending shot into the water as well . '' -lrb- it -rrb- felt good at the time , '' a rueful mcilroy later said of th...",rory mcilroy throws club into water at wgc cadillac championship . northern irishman frustrated after pulling 

In [7]:
# train['article'] = train['article'].apply(lambda s : re.sub('-lrb- cnn -rrb- ', ' ', s))

In [8]:
train.shape

(5000, 2)

In [9]:
train = train.sample(1000)

In [10]:
test = pd.read_csv('/content/cnndm/test.txt.src', sep='\n', header=None)
test.columns=['article']
df2 = pd.read_csv('/content/cnndm/test.txt.tgt.tagged', sep='\n', header=None)
test['highlights'] = df2[0]
test['highlights'] = test['highlights'].apply(lambda s : re.sub('<t>', ' ', s))
test['highlights'] = test['highlights'].apply(lambda s : re.sub('</t>', ' ', s))
test.head()

,article,highlights
0,"marseille , france -lrb- cnn -rrb- the french prosecutor leading an investigation into the crash of germanwings flight 9525 insisted wednesday that he was not aware of any video footage from on board the plane . marseille prosecutor brice robin told cnn that `` so far no videos were used in the crash investigation . '' he added , `` a person who has such a video needs to immediately give it to the investigators . '' robin 's comments follow claims by two magazines , german daily bild and french paris match , of a cell phone video showing the harrowing final seconds from on board germanwing...","marseille prosecutor says `` so far no videos were used in the crash investigation '' despite media reports . journalists at bild and paris match are `` very confident '' the video clip is real , an editor says . andreas lubitz had informed his lufthansa training school of an episode of severe depression , airline says ."
1,"-lrb- cnn -rrb- the palestinian authority officially became the 123rd member of the international criminal court on wednesday , a step that gives the court jurisdiction over alleged crimes in palestinian territories . the formal accession was marked with a ceremony at the hague , in the netherlands , where the court is based . the palestinians signed the icc 's founding rome statute in january , when they also accepted its jurisdiction over alleged crimes committed `` in the occupied palestinian territory , including east jerusalem , since june 13 , 2014 . '' later that month , the icc ope...","membership gives the icc jurisdiction over alleged crimes committed in palestinian territories since last june . israel and the united states opposed the move , which could open the door to war crimes investigations against israelis ."
2,"-lrb- cnn -rrb- governments around the world are using the threat of terrorism -- real or perceived -- to advance executions , amnesty international alleges in its annual report on the death penalty . `` the dark trend of governments using the death penalty in a futile attempt to tackle real or imaginary threats to state security and public safety was stark last year , '' said salil shetty , amnesty 's secretary general in a release . `` it is shameful that so many states around the world are essentially playing with people 's lives -- putting people to death for ` terrorism ' or to quell ...","amnesty 's annual death penalty report catalogs encouraging signs , but setbacks in numbers of those sentenced to death . organization claims that governments around the world are using the threat of terrorism to advance executions . the number of executions worldwide has gone down by almost 22 % compared with 2013 , but death sentences up by 28 % ."
3,"-lrb- cnn -rrb- on may 28 , 2014 , some 7,000 people gathered in a stadium in china 's northwestern xinjiang region . but they had not come to watch the local football team or any other grand sporting event . instead , the authorities paraded scores of prisoners dressed in orange jumpsuits . armed soldiers guarded the exits . in the patently unfair , open air trial that followed , 55 people were found guilty of a range of offenses linked to violent attacks in the region and jailed . three were sentenced to death . the public mass sentencing was part a china 's `` strike hard '' campaign ag...",amnesty international releases its annual review of the death penalty worldwide ; much of it makes for grim reading . salil shetty : countries that use executions to deal with problems are on the wrong side of history .
4,"-lrb- cnn -rrb- seventy years ago , anne frank died of typhus in a nazi concentration camp at the age of 15 . just two weeks after her supposed death on march 31 , 1945 , the bergen-belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the jewish diarist had been to surviving the holocaust . but new research released by the anne frank house shows that anne and her older sist

In [11]:
test.shape

(5000, 2)

In [12]:
train

,article,highlights
2247,"this shocking video shows the moment a pregnant teenager had boiling water thrown over her by her neighbour in a dispute over a garden fence . cynthia almonte lira , 19 , from formosa , northern argentina , was arguing with 63-year-old pensioner celma prado bustos after she built a new fence between their homes . the neighbours can be seen shouting at each other before ms bustos , seen standing in her doorway , throws boiling water at the pregnant woman . distressing : the shocking video shows an elderly woman throwing hot water over a pregnant teenager . screaming in agony , the woman , w...","pregnant teenager and neighbour seen arguing over a garden fence . suddenly the pensioner throws pan of boiling water over the woman , 19 . mother-to-be screams in agony as she tears of her shirt and runs to safety . argentinian woman taken to hospital for burns but her baby was unharmed . the distressing video is being investigated by the police ."
565,"-lrb- cnn -rrb- ferguson 's police chief is gone . so is the city manager . and the top court clerk was fired for sending racist emails . but are these high-level changes enough to satisfy all residents ? absolutely not , they say . protesters amassed in front of the ferguson police department on wednesday night , demanding the entire force be disbanded and calling for mayor james knowles to step down . `` racist cops have got to go , '' some chanted . `` the people , united , will never be defeated , '' others said . amid the protests , two officers were shot -- one in the face , the othe...",two officers are shot in ferguson overnight . police chief thomas jackson 's resignation is effective march 19 . protesters want the entire department gutted and for the mayor to resign .
2904,"it 's kim kardashian 's world - we just play in it . on tuesday kim took to twitter to reveal what her close friend and former editor-at-large for vogue magazine , andre leon talley , would look like in digital form . the flamboyant fashion icon appears as a character on the reality star 's mobile phone game app kim kardashian : hollywood ! , looking suspiciously slimmer than usual . upgrade : andre leon talley is a longtime friend of kim kardashian speaking his mind when it comes to fashion . close : andré leon talley is the former american editor-at-large for vogue magazine . andre will ...","andre leon talley , a former vogue editor , is a colorful and outspoken character . the game is constantly updated with new characters and levels . industry watchers say the app could gross $ 200 million annually and net kardashian a sizable chunk of the game 's profits ."
2258,"this is the moment a father tried to help his police officer son escape after he shot a man during an off-duty bar fight . salome rodriguez , 23 , died from multiple gunshot wounds after henry solis turned on him outside a nightclub in pomona , california , on march 13 . the 27-year-old probationary lapd officer fled the scene to go to his father 's house , an affidavit from an fbi agent claims . victor solis -lrb- left in both pictures -rrb- was charged with making false statements yesterday after cctv footage -lrb- above -rrb- showed him helping his son cross the border into mexico . hen...","victor solis , 53 , was charged with making false statements yesterday . his son henry shot salome rodriguez outside a nightclub on march 13 . 27-year-old probationary officer walked across border just one day later ."
4814,"clarke carlisle pictured outside of court on monday . clarke carlisle has pleaded guilty to a drink-driving offence that occurred two days before the former footballer tried to kill himself by jumping in front of a lorry . the 35-year-old , still bearing the facial scars from his december 22 suicide attempt in a crash which left him in a coma until mid-january , spoke clearly as he stood in the dock at highbury corner magistrates ' court in north london . the former professional footballers ' association c

In [13]:
from datasets import Dataset

dataset = Dataset.from_pandas(train)

In [14]:
dataset

Dataset({
    features: ['article', 'highlights', '__index_level_0__'],
    num_rows: 1000
})

We begin by getting our hugginface objects needed for this task (e.g., the architecture, tokenizer, config, and model).  We'll use blurr's `get_hf_objects` helper method here.

In [15]:
pretrained_model_name = "facebook/bart-large-cnn"
hf_arch, hf_config, hf_tokenizer, hf_model = BLURR.get_hf_objects(pretrained_model_name, 
                                                                  model_cls=BartForConditionalGeneration)

hf_arch, type(hf_config), type(hf_tokenizer), type(hf_model)

('bart',
 transformers.models.bart.configuration_bart.BartConfig,
 transformers.models.bart.tokenization_bart_fast.BartTokenizerFast,
 transformers.models.bart.modeling_bart.BartForConditionalGeneration)

Next we need to build out our DataBlock.  Remember tha a DataBlock is a blueprint describing how to move your raw data into something modelable.  That blueprint is executed when we pass it a data source, which in our case, will be the DataFrame we created above. We'll use a random subset to get things moving along a bit faster for the demo as well.

Notice that the blurr DataBlock as been dramatically simplified given the shift to on-the-fly batch-time tokenization.  All we need is to define a single `HF_Seq2SeqBeforeBatchTransform` instance, optionally passing a list to any of the tokenization arguments to differentiate the values for the input and summary sequences.  In addition to specifying a custom max length for the inputs, we can also do the same for the output sequences ... and with the latest release of blurr, we can even customize the text generation by passing in `text_gen_kwargs`.

We pass `noop` as a type transform for our targets because everything is already handled by the batch transform now.

In [16]:
text_gen_kwargs = default_text_gen_kwargs(hf_config, hf_model, task='summarization'); text_gen_kwargs

{'bad_words_ids': None,
 'bos_token_id': 0,
 'decoder_start_token_id': 2,
 'diversity_penalty': 0.0,
 'do_sample': False,
 'early_stopping': True,
 'encoder_no_repeat_ngram_size': 0,
 'eos_token_id': 2,
 'forced_bos_token_id': 0,
 'forced_eos_token_id': 2,
 'length_penalty': 2.0,
 'max_length': 142,
 'min_length': 56,
 'no_repeat_ngram_size': 3,
 'num_beam_groups': 1,
 'num_beams': 4,
 'num_return_sequences': 1,
 'output_attentions': False,
 'output_hidden_states': False,
 'output_scores': False,
 'pad_token_id': 1,
 'remove_invalid_values': False,
 'repetition_penalty': 1.0,
 'return_dict_in_generate': False,
 'temperature': 1.0,
 'top_k': 50,
 'top_p': 1.0,
 'use_cache': True}

In [17]:
hf_batch_tfm = HF_Seq2SeqBeforeBatchTransform(hf_arch, hf_config, hf_tokenizer, hf_model, 
                                              max_length=256, max_tgt_length=130, text_gen_kwargs=text_gen_kwargs)

blocks = (HF_Seq2SeqBlock(before_batch_tfm=hf_batch_tfm), noop)

dblock = DataBlock(blocks=blocks, get_x=ColReader('article'), get_y=ColReader('highlights'), splitter=RandomSplitter())

In [18]:
dls = dblock.dataloaders(train, bs=2)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")


In [19]:
len(dls.train.items), len(dls.valid.items)

(800, 200)

It's always a good idea to check out a batch of data and make sure the shapes look right.

In [20]:
b = dls.one_batch()
len(b), b[0]['input_ids'].shape, b[1].shape

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")


(2, torch.Size([2, 256]), torch.Size([2, 96]))

Even better, we can take advantage of blurr's TypeDispatched version of `show_batch` to look at things a bit more intuitively.  We pass in the `dls` via the `dataloaders` argument so we can access all tokenization/modeling configuration stored in our batch transform above.

In [21]:
dls.show_batch(dataloaders=dls, max_n=2)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")


,text,target
0,"<s> turn on the telly, lie back and think of england the way it used to be. it isn't difficult. not with david james, phil neville, jamie carragher, martin keown and lee dixon sharing their expertise and defensive acumen. or maybe there's the midfield craft of steve mcmanaman, paul scholes, owen hargreaves and jamie redknapp. or the striking instincts of alan shearer and michael owen. that's a decent international team right there, boasting more than 500 caps and more than a hundred goals, even without match of the day host gary lineker. or gary neville, one of a few england internationals from his generation involved in coaching, when his sky tv duties allow. holland beat england 2-0 in 1993 and almost all of the dutch side are in coaching but not one player from england is in management... holland. de goey : now works with goalkeepers at dutch amateur side dhc delft. de wolf : ex-boss at zwart, halsteren, svvsmc, haaglandia, turkiyemspor</s>","in 1993, holland beat england 2-0 in a world cup qualifier. many of the england internationals from that night are now working in punditry. on the holland side however, most players are now working in coaching. only four former england players are now managing across the top four leagues."
1,"<s> at 100-plus kilometres per hour, molly taylor's car goes into a sideways slide that has it teetering on the brink of flipping over into an uncontrolled roll. no problem for ms taylor. she adjusts the steering wheel, shift's down a gear, and slams her foot on the accelerator to power out of the turn. meet the young australian driver who is smashing through the glass ceiling of motorsports. not literally, of course. australian molly taylor is the only woman competing in the world rally championships. motorsports has always traditionally been a macho industry, but two women are breaking through the glass ceiling with fearless grit and determination. ms taylor, 26, is the only woman competing on the world rally championship circuit - a competition in which 19 drivers have been killed since it began in the earlier 1970s. she has joined brit susie wolff, the only female formula one test driver in the world, as women at the top of their chosen sports, both traditionally dominate by men. both women were born with racing pumping through their veins - their parents worked in the motorsports arena. although it wasn't expected that they follow in their footsteps - they were certainly never discouraged to take up</s>","motorsports have traditionally been male dominated, but not any more. australian molly taylor the only woman in the world rally championships. brit susie wolff is the only woman formula one test driver. daily mail australia gained exclusive access before melbourne grand prix."


## Training

We'll prepare our BART model for training by wrapping it in blurr's `HF_BaseModelWrapper` object and using the callback, `HF_BaseModelCallback`, as usual.  A new `HF_Seq2SeqMetricsCallback` object allows us to specify Seq2Seq metrics we want to use, things like rouge and bertscore for tasks like summarization as well as metrics such as meteor, bleu, and sacrebleu for translations tasks. Using huggingface's metrics library is as easy as specifying a metrics configuration such as below.

Once we have everything in place, we'll freeze our model so that only the last layer group's parameters of trainable.  See [here](https://docs.fast.ai/basic_train.html#Discriminative-layer-training) for our discriminitative learning rates work in fastai.

**Note:** This has been tested with ALOT of other Seq2Seq models; see the docs for more information.

In [22]:
seq2seq_metrics = {
        'rouge': {
            'compute_kwargs': { 'rouge_types': ["rouge1", "rouge2", "rougeL"], 'use_stemmer': True },
            'returns': ["rouge1", "rouge2", "rougeL"]
        },
        'bertscore': {
            'compute_kwargs': { 'lang': 'en' },
            'returns': ["precision", "recall", "f1"]
        },
        'bleu': { 'returns': "bleu" },
    }

In [23]:
model = HF_BaseModelWrapper(hf_model)
learn_cbs = [HF_BaseModelCallback]
fit_cbs = [HF_Seq2SeqMetricsCallback(custom_metrics=seq2seq_metrics)]

learn = Learner(dls, 
                model,
                opt_func=ranger,
                loss_func=CrossEntropyLossFlat(),
                cbs=learn_cbs,
                splitter=partial(seq2seq_splitter, arch=hf_arch)).to_fp16()

learn.create_opt() 
learn.freeze()

Still experimenting with how to use fastai's learning rate finder for these kinds of models.  If you all have any suggestions or interesting insights to share, please let me know.  We're only going to train the frozen model for one epoch for this demo, but feel free to progressively unfreeze the model and train the other layers to see if you can best my results below.

In [24]:
# learn.lr_find()

It's also not a bad idea to run a batch through your model and make sure the shape of what goes in, and comes out, looks right.

In [25]:
b = dls.one_batch()
preds = learn.model(b[0])
len(preds),preds[0], preds[1].shape

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")


(4,
 tensor(3.5467, device='cuda:0', grad_fn=<NllLossBackward>),
 torch.Size([2, 84, 50264]))

In [27]:
learn.fit_one_cycle(5, lr_max=3e-4, cbs=fit_cbs)

epoch,train_loss,valid_loss,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,bleu,time
0,1.425233,1.654672,0.399944,0.180374,0.276150,0.882647,0.877572,0.879968,0.209781,13:05
1,1.307939,1.572839,0.380265,0.161925,0.253578,0.873064,0.881103,0.876891,0.213337,14:26
2,0.800243,1.759768,0.378951,0.153231,0.243312,0.870362,0.878669,0.874324,0.188796,14:57
3,0.386339,1.900798,0.383552,0.158148,0.247747,0.869221,0.885928,0.877333,0.190381,15:37
4,0.196882,2.248372,0.382402,0.156472,0.246569,0.871493,0.883834,0.877486,0.198346,14:45


/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")
/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")
/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")
/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.warn("`max_length` is ignored when `padding`=`True`.")
/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2227: UserWarning: `max_length` is ignored when `padding`=`True`.
  warnings.

And now we can look at the generated predictions using our `text_gen_kwargs` above

In [ ]:
learn.show_results(learner=learn, max_n=2)

Even better though, blurr augments the fastai Learner with a `blurr_summarize` method that allows you to use huggingface's `PreTrainedModel.generate` method to create something more human-like.

In [ ]:
test_article = """
The past 12 months have been the worst for aviation fatalities so far this decade - with the total of number of people killed if airline 
crashes reaching 1,050 even before the Air Asia plane vanished. Two incidents involving Malaysia Airlines planes - one over eastern Ukraine and the other in the Indian Ocean - led to the deaths of 537 people, while an Air Algerie crash in Mali killed 116 and TransAsia Airways crash in Taiwan killed a further 49 people. The remaining 456 fatalities were largely in incidents involving small commercial planes or private aircraft operating on behalf of companies, governments or organisations. Despite 2014 having the highest number of fatalities so far this decade, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949 - totalling just 111 across the whole world over the past 12 months. The all-time deadliest year for aviation was 1972 when a staggering 2,429 people were killed in a total of 55 plane crashes - including the crash of Aeroflot Flight 217, which killed 174 people in Russia, and Convair 990 Coronado, which claimed 155 lives in Spain. However this year's total death count of 1,212, including those presumed dead on board the missing Air Asia flight, marks a significant rise on the very low 265 fatalities in 2013 - which led to it being named the safest year in aviation since the end of the Second World War. Scroll down for videos. Deadly: The past 12 months have been the worst for aviation fatalities so far this decade - with the total of number of people killed if airline crashes reaching 1,158 even before the Air Asia plane (pictured) vanished. Fatal: Two incidents involving Malaysia Airlines planes - one over eastern Ukraine (pictured) and the other in the Indian Ocean - led to the deaths of 537 people. Surprising: Despite 2014 having the highest number of fatalities so far this decade, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949. 2014 has been a horrific year for Malaysia-based airlines, with 537 people dying on Malaysia Airlines planes, and a further 162 people missing and feared dead in this week's Air Asia incident. In total more than half the people killed in aviation incidents this year had been flying on board Malaysia-registered planes. In January a total of 12 people lost their lives in five separate incidents, while the same number of crashes in February killed 107. 
"""

We can override the `text_gen_kwargs` we specified for our `DataLoaders` when we generate text using blurr's `Learner.blurr_generate` method

In [ ]:
outputs = learn.blurr_generate(test_article, early_stopping=True, num_beams=4, num_return_sequences=3)

for idx, o in enumerate(outputs):
    print(f'=== Prediction {idx+1} ===\n{o}\n')

What about inference?  Easy!

In [ ]:
learn.metrics = None
learn.export(fname='ft_cnndm_export.pkl')

In [ ]:
inf_learn = load_learner(fname='ft_cnndm_export.pkl')
inf_learn.blurr_generate(test_article)